# Hybrid Bengali Hallucination Detector — অলীকবচন

Two-route detector (see `plans.md`). The test set is routed by `has_context`:

* **context rows** → Route A: retrieval → Bengali extractive QA → heuristics → LR
* **no-context rows** → Route B: closed-book KB retrieval-verifier → LR

This notebook **materializes the `hallu/` package** (via `%%writefile`), then runs
it end-to-end to write `/kaggle/working/submission.csv`.

## Attach these as Kaggle datasets
1. **Test files** — `test set.csv`, `dataset samples.json` (the competition data).
2. **Knowledge base** — `combined_bengali_qa.csv` (~157k Bengali QA pairs).
3. *(optional, for offline)* prebuilt indices dir (`kb_lexical_indices.pkl`,
   `labse_kb.npy`, `labse_kb_ids.npy`), the QA model, and LaBSE weights.

Paths are auto-detected from `/kaggle/input` in the **Config** cell below —
adjust there if detection misses. Everything transformer-based is optional and
falls back to a pure-lexical mode, so the notebook runs even with no models
attached.

## 1. Dependencies
`rank_bm25` and `rapidfuzz` are not preinstalled on Kaggle; the rest usually are.

In [ ]:
import subprocess, sys
def _pip(*pkgs):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    except Exception as e:
        print("pip install failed (offline?):", e)
_pip("rank_bm25", "rapidfuzz")
# sklearn / joblib / transformers / sentence-transformers ship with Kaggle images.


## 2. Materialize the `hallu/` package

In [ ]:
import os
os.makedirs('hallu', exist_ok=True)
print('created hallu/')

In [ ]:
%%writefile hallu/__init__.py
"""Hybrid Bengali hallucination detector (context + no-context routing).

Package layout (mirrors plans.md build order):
    normalize.py  -- shared preprocessing / normalizer / number parsing (S3)
    kb.py         -- knowledge-base load + clean + dedupe (S1)
    features.py   -- shared answer-comparison features (heuristic cell 28) (S4.3)
    route_a.py    -- contextual verifier, predict_context_rows() (S2)
    indices.py    -- KB indices: inverted Q/A, NER, BM25, LaBSE (S4.1)
    route_b.py    -- no-context retrieval verifier (S4.2-4.4)
    pipeline.py   -- assembly + submission writer (S6)
    validate.py   -- validation / weight+threshold tuning on sample split (S5)

Every transformer-dependent signal (QA model, LaBSE, NER model) is optional and
degrades to a pure-lexical mode so the whole pipeline runs offline.
"""

__all__ = ["normalize", "kb", "features", "route_a", "indices", "route_b", "pipeline"]


In [ ]:
%%writefile hallu/normalize.py
"""Shared preprocessing / normalizer layer (plans.md S3).

One canonical normalizer used by the KB indexer, both routes, and query time so
that inverted-index keys and comparison features live in the same space.

The number-word / ordinal machinery is ported verbatim from
``heuristic_qa.ipynb`` cell 6 (it is battle-tested and the two routes must agree
on numeric equality), with the sentence/tokenise helpers from cells 10 and 17.
"""

import re
import unicodedata
from collections import Counter

# ---------------------------------------------------------------------------
# Null / context detection (drives the A/B router)
# ---------------------------------------------------------------------------
NO_CONTEXT = {"", "nan", "NaN", "[NULL]", "null", "None", "none", "NULL"}


def clean_context(value):
    """Return a stripped context string, or "" for any null-ish marker."""
    if value is None:
        return ""
    text = str(value).strip()
    if text in NO_CONTEXT:
        return ""
    return text


def has_context(value):
    """True when the row carries a usable supporting passage."""
    return len(clean_context(value)) > 0


# ---------------------------------------------------------------------------
# Bengali -> ASCII digit folding
# ---------------------------------------------------------------------------
BN_DIGITS = "০১২৩৪৫৬৭৮৯"
BN2ASCII = str.maketrans(BN_DIGITS, "0123456789")

STOPWORDS = set(
    "এর এই এবং ছিল হয় সালে ইত্যাদি করে হয়েছে থেকে জন্য কে কি কী ও তার তাঁর তিনি ছিলেন "
    "হয়েছিল করেন হলো ছিলো একটি একজন যা যে দিয়ে সহ পর প্রথম মধ্যে অন্তর্গত নামে বলা কত কোন "
    "কোথায় কোথায় সঙ্গে রয়েছে করা হচ্ছে তখন এতে সেই বছর সাল খ্রিস্টাব্দে খ্রিষ্টাব্দে".split()
)

UNITS = {
    "শূন্য": 0, "এক": 1, "দুই": 2, "তিন": 3, "চার": 4, "পাঁচ": 5,
    "ছয়": 6, "সাত": 7, "আট": 8, "নয়": 9,
    "দশ": 10, "এগারো": 11, "বারো": 12, "তেরো": 13, "চৌদ্দ": 14,
    "পনেরো": 15, "ষোল": 16, "ষোলো": 16, "সতেরো": 17,
    "আঠারো": 18, "উনিশ": 19, "ঊনিশ": 19, "বিশ": 20,
    "একুশ": 21, "বাইশ": 22, "তেইশ": 23, "চব্বিশ": 24,
    "পঁচিশ": 25, "ছাব্বিশ": 26, "সাতাশ": 27, "আটাশ": 28,
    "ঊনত্রিশ": 29, "উনত্রিশ": 29, "ত্রিশ": 30, "একত্রিশ": 31,
    "বত্রিশ": 32, "তেত্রিশ": 33, "চৌত্রিশ": 34,
    "পঁয়ত্রিশ": 35, "ছত্রিশ": 36,
    "সাঁইত্রিশ": 37, "আটত্রিশ": 38, "ঊনচল্লিশ": 39,
    "উনচল্লিশ": 39, "চল্লিশ": 40, "একচল্লিশ": 41,
    "বিয়াল্লিশ": 42, "তেতাল্লিশ": 43,
    "চুয়াল্লিশ": 44, "পঁয়তাল্লিশ": 45,
    "ছেচল্লিশ": 46, "সাতচল্লিশ": 47,
    "আটচল্লিশ": 48, "ঊনপঞ্চাশ": 49, "উনপঞ্চাশ": 49,
    "পঞ্চাশ": 50, "একান্ন": 51, "বায়ান্ন": 52,
    "তেপ্পান্ন": 53, "চুয়ান্ন": 54,
    "পঞ্চান্ন": 55, "ছাপ্পান্ন": 56, "সাতান্ন": 57,
    "আটান্ন": 58, "ঊনষাট": 59, "উনষাট": 59, "ষাট": 60,
    "একষট্টি": 61, "বাষট্টি": 62, "তেষট্টি": 63,
    "চৌষট্টি": 64, "পঁয়ষট্টি": 65,
    "ছেষট্টি": 66, "সাতষট্টি": 67, "আটষট্টি": 68,
    "ঊনসত্তর": 69, "উনসত্তর": 69, "সত্তর": 70,
    "একাত্তর": 71, "বাহাত্তর": 72, "তিয়াত্তর": 73,
    "চুয়াত্তর": 74,
    "পঁচাত্তর": 75, "ছিয়াত্তর": 76,
    "সাতাত্তর": 77, "আটাত্তর": 78, "ঊনআশি": 79,
    "উনআশি": 79, "আশি": 80, "একাশি": 81, "বিরাশি": 82,
    "তিরাশি": 83, "চুরাশি": 84, "পঁচাশি": 85,
    "ছিয়াশি": 86, "সাতাশি": 87,
    "আটাশি": 88, "ঊননব্বই": 89, "উননব্বই": 89,
    "নব্বই": 90, "একানব্বই": 91, "বিরানব্বই": 92,
    "তিরানব্বই": 93, "চুরানব্বই": 94, "পঁচানব্বই": 95,
    "ছিয়ানব্বই": 96, "সাতানব্বই": 97,
    "আটানব্বই": 98, "নিরানব্বই": 99,
}

RESOLVED_HUNDREDS = {
    "একশ": 100, "একশত": 100, "দুইশ": 200, "দুশ": 200, "দুইশত": 200,
    "তিনশ": 300, "তিনশত": 300, "চারশ": 400, "চারশত": 400,
    "পাঁচশ": 500, "পাঁচশত": 500, "ছয়শ": 600,
    "ছয়শত": 600, "সাতশ": 700, "সাতশত": 700,
    "আটশ": 800, "আটশত": 800, "নয়শ": 900,
    "নয়শত": 900, "এগারোশ": 1100,
    "বারোশ": 1200, "তেরোশ": 1300, "চৌদ্দশ": 1400,
    "পনেরোশ": 1500, "ষোলশ": 1600, "ষোলোশ": 1600,
    "সতেরশ": 1700, "সতেরোশ": 1700, "আঠারশ": 1800,
    "আঠারোশ": 1800, "উনিশশ": 1900, "ঊনিশশ": 1900,
}

SMALL_MULTIPLIERS = {"শ": 100, "শত": 100, "hundred": 100, "hundreds": 100}

LARGE_MULTIPLIERS = {
    "হাজার": 1_000, "লক্ষ": 100_000, "লাখ": 100_000, "কোটি": 10_000_000,
    "মিলিয়ন": 1_000_000, "বিলিয়ন": 1_000_000_000, "ট্রিলিয়ন": 1_000_000_000_000,
    "thousand": 1_000, "thousands": 1_000, "million": 1_000_000, "millions": 1_000_000,
    "billion": 1_000_000_000, "billions": 1_000_000_000,
    "trillion": 1_000_000_000_000, "trillions": 1_000_000_000_000,
}

NUMBER_SUFFIXES = tuple(
    sorted(
        ("গুলো", "গুলোর", "টিকে", "টির", "দের", "খানা", "খানি",
         "জন", "তম", "টার", "টা", "টি", "এর", "র"),
        key=len,
        reverse=True,
    )
)

NUMBER_CONNECTORS = {"ও", "এবং", "and"}

PUNCTUATION_RE = re.compile(r"""["'।,.?!()\[\]{}‘’“”:;—–\-]""")
# "." kept out of the numeric variant because it can be a decimal point.
NUMBER_PUNCTUATION_RE = re.compile(r"""["'।,?!()\[\]{}‘’“”:;—–\-]""")


def normalize_text(text):
    """Unicode NFKC, strip zero-width joiners/BOM, collapse whitespace."""
    text = unicodedata.normalize("NFKC", str(text or ""))
    text = text.replace("‌", "").replace("‍", "").replace("﻿", "")
    return re.sub(r"\s+", " ", text).strip()


def normalize_answer_text(text):
    """Canonical comparison form: normalized + digit-folded + lower + depunct."""
    text = normalize_text(text).translate(BN2ASCII).lower()
    text = PUNCTUATION_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text):
    """Content tokens (length > 1, non-stopword) over the canonical form."""
    return [
        token
        for token in normalize_answer_text(text).split()
        if len(token) > 1 and token not in STOPWORDS
    ]


def normalize_numeric_string(value):
    try:
        value = float(value)
        if value.is_integer():
            return str(int(value))
        return f"{value:.10f}".rstrip("0").rstrip(".")
    except (TypeError, ValueError):
        return str(value)


def normalize_number_word_suffix(token):
    """Strip attached counters/grammatical suffixes: চারটি->চার, ৪২টি->42."""
    token = str(token).strip().lower()
    number_vocab = (
        set(UNITS) | set(RESOLVED_HUNDREDS) | set(SMALL_MULTIPLIERS) | set(LARGE_MULTIPLIERS)
    )
    if re.fullmatch(r"\d+(?:\.\d+)?", token):
        return token
    if token in number_vocab:
        return token
    for suffix in NUMBER_SUFFIXES:
        if not token.endswith(suffix):
            continue
        root = token[: -len(suffix)]
        if re.fullmatch(r"\d+(?:\.\d+)?", root) or root in number_vocab:
            return root
    return token


def token_numeric_value(token):
    token = normalize_number_word_suffix(token)
    if re.fullmatch(r"\d+(?:\.\d+)?", token):
        return float(token)
    if token in UNITS:
        return float(UNITS[token])
    if token in RESOLVED_HUNDREDS:
        return float(RESOLVED_HUNDREDS[token])
    return None


def is_numeric_phrase_token(token):
    token = normalize_number_word_suffix(token)
    return (
        token_numeric_value(token) is not None
        or token in SMALL_MULTIPLIERS
        or token in LARGE_MULTIPLIERS
        or token in NUMBER_CONNECTORS
    )


def parse_numeric_phrase(tokens):
    """Parse a contiguous numeric phrase (handles nested scales recursively)."""
    normalized_tokens = []
    for token in tokens:
        token = normalize_number_word_suffix(token)
        if token not in NUMBER_CONNECTORS:
            normalized_tokens.append(token)
    tokens = normalized_tokens
    if not tokens:
        return None

    scale_positions = [
        (index, LARGE_MULTIPLIERS[token])
        for index, token in enumerate(tokens)
        if token in LARGE_MULTIPLIERS
    ]
    if scale_positions:
        largest_scale = max(scale for _, scale in scale_positions)
        scale_index = next(
            index for index, scale in scale_positions if scale == largest_scale
        )
        left_value = parse_numeric_phrase(tokens[:scale_index])
        right_value = parse_numeric_phrase(tokens[scale_index + 1:])
        if left_value is None:
            left_value = 1.0
        if right_value is None:
            right_value = 0.0
        return left_value * largest_scale + right_value

    current = 0.0
    found_number = False
    for token in tokens:
        numeric_value = token_numeric_value(token)
        if numeric_value is not None:
            current += numeric_value
            found_number = True
            continue
        if token in SMALL_MULTIPLIERS:
            current = max(current, 1.0) * SMALL_MULTIPLIERS[token]
            found_number = True
            continue
        return None
    return current if found_number else None


def _prepare_numeric_text(text):
    text = normalize_text(text).translate(BN2ASCII).lower()
    text = re.sub(r"(?<=\d),(?=\d)", "", text)  # 1,000,000 -> 1000000
    text = re.sub(r"(\d+)\s*দশমিক\s*(\d+)", r"\1.\2", text)  # ৭ দশমিক ৬৩ -> 7.63
    text = re.sub(r"(?<!\d)\.|\.(?!\d)", " ", text)  # keep 7.63, drop "2018."
    text = NUMBER_PUNCTUATION_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def _extract_numeric_values(text, include_digit_tokens=True):
    prepared_text = _prepare_numeric_text(text)
    tokens = prepared_text.split()
    found = set()
    index = 0
    while index < len(tokens):
        first_token = normalize_number_word_suffix(tokens[index])
        first_is_digit = bool(re.fullmatch(r"\d+(?:\.\d+)?", first_token))
        first_is_numeric = token_numeric_value(first_token) is not None
        first_is_multiplier = (
            first_token in SMALL_MULTIPLIERS or first_token in LARGE_MULTIPLIERS
        )
        if not first_is_numeric and not first_is_multiplier:
            index += 1
            continue
        if first_is_digit and not include_digit_tokens:
            index += 1
            continue

        phrase_tokens = []
        next_index = index
        while next_index < len(tokens):
            token = normalize_number_word_suffix(tokens[next_index])
            token_is_digit = bool(re.fullmatch(r"\d+(?:\.\d+)?", token))
            if token in NUMBER_CONNECTORS:
                if (
                    phrase_tokens
                    and next_index + 1 < len(tokens)
                    and is_numeric_phrase_token(tokens[next_index + 1])
                ):
                    phrase_tokens.append(token)
                    next_index += 1
                    continue
                break
            if not is_numeric_phrase_token(token):
                break
            if token_is_digit and not include_digit_tokens:
                break
            phrase_tokens.append(token)
            next_index += 1

        contains_multiplier = any(
            token in SMALL_MULTIPLIERS or token in LARGE_MULTIPLIERS
            for token in phrase_tokens
        )
        contains_word_number = any(
            token in UNITS or token in RESOLVED_HUNDREDS for token in phrase_tokens
        )
        digit_tokens = [
            token for token in phrase_tokens if re.fullmatch(r"\d+(?:\.\d+)?", token)
        ]
        if contains_multiplier or contains_word_number:
            value = parse_numeric_phrase(phrase_tokens)
            if value is not None:
                found.add(normalize_numeric_string(value))
        else:
            for token in digit_tokens:
                found.add(normalize_numeric_string(token))
        index = max(next_index, index + 1)
    return found


def parse_number_words(text):
    """Bengali/English number-word expressions, excluding bare digit values."""
    return _extract_numeric_values(text, include_digit_tokens=False)


def extract_numbers(text):
    """Normalized numeric values (digits + scaled word quantities)."""
    return _extract_numeric_values(text, include_digit_tokens=True)


# ---------------------------------------------------------------------------
# Ordinal handling (heuristic cell 22)
# ---------------------------------------------------------------------------
ORDINALS = {
    "প্রথম": 1, "দ্বিতীয়": 2, "তৃতীয়": 3, "চতুর্থ": 4, "পঞ্চম": 5,
    "ষষ্ঠ": 6, "সপ্তম": 7, "অষ্টম": 8, "নবম": 9, "দশম": 10,
    "একাদশ": 11, "দ্বাদশ": 12, "ত্রয়োদশ": 13, "চতুর্দশ": 14,
    "পঞ্চদশ": 15, "ষোড়শ": 16, "সপ্তদশ": 17, "অষ্টাদশ": 18,
    "ঊনবিংশ": 19, "উনবিংশ": 19, "বিংশ": 20,
}


def extract_ordinals(text):
    normalized = normalize_text(text).translate(BN2ASCII).lower()
    values = {
        value
        for word, value in ORDINALS.items()
        if re.search(rf"(?<!\w){re.escape(word)}(?!\w)", normalized)
    }
    digit_ordinals = re.findall(r"(\d+)\s*(?:ম|তম|st|nd|rd|th)\b", normalized)
    values.update(int(value) for value in digit_ordinals)
    return values


def integer_numbers(text):
    values = set()
    for value in extract_numbers(text):
        try:
            numeric = float(value)
            if numeric.is_integer():
                values.add(int(numeric))
        except (TypeError, ValueError):
            continue
    return values


# ---------------------------------------------------------------------------
# Sentence splitting + token F1 (heuristic cells 10, 17)
# ---------------------------------------------------------------------------
def split_sentences(context):
    raw = normalize_text(context)
    sentences = re.split(r"(?<=[।!?])\s+|\n+", raw)
    sentences = [
        normalize_text(sentence)
        for sentence in sentences
        if len(normalize_text(sentence)) >= 5
    ]
    return sentences or [raw]


def token_f1(a, b):
    a_tokens = normalize_answer_text(a).split()
    b_tokens = normalize_answer_text(b).split()
    if not a_tokens or not b_tokens:
        return 0.0
    ac, bc = Counter(a_tokens), Counter(b_tokens)
    common = sum(min(ac[t], bc[t]) for t in set(ac) | set(bc))
    precision = common / len(b_tokens)
    recall = common / len(a_tokens)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


In [ ]:
%%writefile hallu/kb.py
"""Knowledge-base load + clean + dedupe (plans.md S1).

`combined_bengali_qa.csv` -- ~157k Bengali (question, gold_answer) units used as
the retrieval corpus for route B.  Every KB answer is treated as gold/faithful:
a test response that agrees with the retrieved answer -> label 1, contradicts
-> label 0.
"""

import os

import pandas as pd

from .normalize import normalize_answer_text, normalize_text


def load_kb(csv_path, cache_path=None, sample_questions=None):
    """Load, clean and dedupe the KB.

    Parameters
    ----------
    csv_path : str
        Path to ``combined_bengali_qa.csv``.
    cache_path : str, optional
        Pickle cache; built on first call, reused afterwards.
    sample_questions : iterable[str], optional
        Normalized questions from the labelled sample split to hold out, so no
        KB unit that is literally a validation row can leak in (plans.md S5).

    Returns
    -------
    pandas.DataFrame with columns
        kb_id, prompt_bn, response_bn, source,
        q_norm (normalized question), a_norm (normalized answer)
    """
    if cache_path and os.path.exists(cache_path):
        kb = pd.read_pickle(cache_path)
    else:
        kb = pd.read_csv(csv_path)
        for col in ("prompt_bn", "response_bn"):
            kb[col] = kb[col].fillna("").astype(str)
        if "source" not in kb.columns:
            kb["source"] = ""
        kb["source"] = kb["source"].fillna("").astype(str)

        # 1. Drop empty question/answer rows.
        kb = kb[kb["prompt_bn"].str.strip().ne("")]
        kb = kb[kb["response_bn"].str.strip().ne("")]

        # 2. Normalize (shared normalizer -- same space as query time).
        kb["q_norm"] = kb["prompt_bn"].map(normalize_answer_text)
        kb["a_norm"] = kb["response_bn"].map(normalize_answer_text)
        kb = kb[kb["q_norm"].ne("") & kb["a_norm"].ne("")]

        # 3. Deduplicate on normalized (question, answer).
        kb = kb.drop_duplicates(subset=["q_norm", "a_norm"]).reset_index(drop=True)

        kb = kb[["prompt_bn", "response_bn", "source", "q_norm", "a_norm"]].copy()
        if cache_path:
            os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
            kb.to_pickle(cache_path)

    # 4. Hold out validation questions (leakage guard) -- applied post-cache so
    #    the same cache serves both the tuning and the submission runs.
    if sample_questions:
        held = set(normalize_answer_text(q) for q in sample_questions)
        before = len(kb)
        kb = kb[~kb["q_norm"].isin(held)].copy()
        dropped = before - len(kb)
        if dropped:
            print(f"[kb] leakage guard: dropped {dropped} KB rows matching sample questions")

    kb = kb.reset_index(drop=True)
    kb["kb_id"] = kb.index.to_numpy()
    return kb


In [ ]:
%%writefile hallu/features.py
"""Shared answer-comparison features (plans.md S4.3, heuristic cells 17/22/26/28).

In route A the second argument is the extractive-QA answer; in route B it is the
retrieved KB gold answer.  Same machinery, same feature names -- so a single
calibrator recipe applies to both routes.
"""

import re

from rapidfuzz.fuzz import ratio, token_set_ratio

from .normalize import (
    extract_numbers,
    extract_ordinals,
    integer_numbers,
    normalize_answer_text,
    normalize_text,
    tokenize,
    token_f1,
)

YES_WORDS = {"হ্যাঁ", "হ্যা", "জি", "জ্বি", "সত্য"}
NO_WORDS = {"না", "নয়", "মিথ্যা"}

RELATION_SENSITIVE_CUES = (
    "প্রথম", "সর্বপ্রথম", "শেষ", "সর্বশেষ", "বর্তমান",
    "একমাত্র", "প্রধান", "সবচেয়ে",
)


# ---------------------------------------------------------------------------
# Answer-type expectations (cell 17)
# ---------------------------------------------------------------------------
def expected_answer_type(prompt):
    p = normalize_text(prompt).lower()
    if any(x in p for x in ("কত সালে", "কোন সালে", "জন্মসাল", "কবে")):
        return "date"
    if any(
        x in p
        for x in ("কতটি", "কয়টি", "কতগুলি", "উচ্চতা কত", "দৈর্ঘ্য কত",
                  "বয়স কত", "জনসংখ্যা কত", "কত শতাংশ", "দূরত্ব কত")
    ):
        return "number"
    if any(
        x in p
        for x in ("কে?", "কে ", "কার নাম", "নাম কী", "নাম কি",
                  "রচয়িতা", "লেখক কে", "প্রতিষ্ঠাতা কে")
    ):
        return "entity"
    if any(x in p for x in ("কোথায়", "কোন দেশে", "কোন জেলায়")):
        return "location"
    if "কেন" in p:
        return "reason"
    return "other"


def detected_response_type(response):
    r = normalize_text(response).lower()
    if r in YES_WORDS | NO_WORDS:
        return "yes_no"
    if extract_numbers(response):
        return "number"
    return "short_entity" if len(tokenize(response)) <= 5 else "sentence"


def answer_type_mismatch(prompt, response):
    expected = expected_answer_type(prompt)
    actual = detected_response_type(response)
    if expected in {"date", "number"} and actual != "number":
        return 1
    if expected in {"entity", "location"} and actual in {"number", "yes_no"}:
        return 1
    if expected == "reason" and actual in {"number", "yes_no"}:
        return 1
    return 0


def candidate_wrong_type_for_numeric_question(prompt, candidate):
    """1 when the question needs a number/date but the candidate carries none."""
    numeric_question = expected_answer_type(prompt) in {"number", "date"}
    candidate_has_number = bool(extract_numbers(candidate))
    return int(numeric_question and not candidate_has_number)


def has_relation_sensitive_modifier(prompt):
    prompt_normalized = normalize_text(prompt).lower()
    return int(any(cue in prompt_normalized for cue in RELATION_SENSITIVE_CUES))


def qa_rank_to_count_agreement(prompt, candidate, qa_answer):
    """Reconcile rank<->count: rank 8 => 7 ahead (cell 22)."""
    prompt_normalized = normalize_text(prompt).lower()
    count_cues = ("কয়টি", "কতটি", "কতগুলি", "কতগুলো", "কত জন")
    comparison_cues = (
        "থেকেও", "থেকে বেশি", "চেয়ে", "আগে রয়েছে",
        "এর আগে", "উপরে রয়েছে",
    )
    asks_for_count = any(cue in prompt_normalized for cue in count_cues)
    asks_relative_rank = any(cue in prompt_normalized for cue in comparison_cues)
    if not (asks_for_count and asks_relative_rank):
        return 0
    candidate_values = integer_numbers(candidate)
    qa_ranks = extract_ordinals(qa_answer)
    return int(any(rank >= 1 and (rank - 1) in candidate_values for rank in qa_ranks))


def candidate_qa_same_sentence(context, candidate, qa_answer):
    """1 when candidate and gold answer co-occur in one context sentence."""
    sentences = re.split(r"(?<=[।!?])\s+|\n+", normalize_text(context))
    candidate_normalized = normalize_answer_text(candidate)
    qa_normalized = normalize_answer_text(qa_answer)
    if not candidate_normalized or not qa_normalized:
        return 0
    for sentence in sentences:
        sentence_normalized = normalize_answer_text(sentence)
        if candidate_normalized in sentence_normalized and qa_normalized in sentence_normalized:
            return 1
    return 0


# ---------------------------------------------------------------------------
# Core response<->gold comparison (cell 28)
# ---------------------------------------------------------------------------
def qa_comparison_features(candidate, qa_answer):
    cn = normalize_answer_text(candidate)
    qn = normalize_answer_text(qa_answer)

    candidate_numbers = extract_numbers(candidate)
    qa_numbers = extract_numbers(qa_answer)
    both_numeric = bool(candidate_numbers and qa_numbers)

    numeric_exact = int(both_numeric and candidate_numbers == qa_numbers)
    numeric_overlap = (
        len(candidate_numbers & qa_numbers) / len(candidate_numbers | qa_numbers)
        if both_numeric
        else 0.0
    )
    numeric_conflict = int(both_numeric and not bool(candidate_numbers & qa_numbers))

    text_exact = int(bool(cn) and cn == qn)
    candidate_in_qa = int(bool(cn) and cn in qn)
    qa_in_candidate = int(bool(qn) and qn in cn)
    f1 = token_f1(candidate, qa_answer)
    token_set = token_set_ratio(cn, qn) / 100.0
    char_sim = ratio(cn, qn) / 100.0

    agreement = max(
        float(numeric_exact), float(text_exact),
        float(candidate_in_qa), float(qa_in_candidate),
        f1, token_set, char_sim,
    )

    return {
        "qa_both_have_numbers": int(both_numeric),
        "qa_numeric_exact_match": numeric_exact,
        "qa_numeric_overlap": numeric_overlap,
        "qa_numeric_conflict": numeric_conflict,
        "qa_text_exact_match": text_exact,
        "qa_candidate_in_answer": candidate_in_qa,
        "qa_answer_in_candidate": qa_in_candidate,
        "qa_token_f1": f1,
        "qa_token_set_similarity": token_set,
        "qa_character_similarity": char_sim,
        "qa_normalized_agreement": agreement,
    }


def build_comparison_features(prompt, response, gold_answer, context=""):
    """Full comparison-feature dict between a test response and a gold answer.

    Mirrors the per-row assembly of heuristic cell 28 (minus the QA-score
    columns, which each route appends itself since they mean different things).
    """
    features = qa_comparison_features(candidate=response, qa_answer=gold_answer)

    wrong_numeric_type = candidate_wrong_type_for_numeric_question(prompt, response)
    rank_agreement = qa_rank_to_count_agreement(prompt, response, gold_answer)
    relation_sensitive = has_relation_sensitive_modifier(prompt)
    same_sentence = candidate_qa_same_sentence(context, response, gold_answer)

    features["candidate_wrong_type_for_numeric_question"] = int(wrong_numeric_type)
    features["qa_rank_to_count_agreement"] = int(rank_agreement)
    features["question_has_relation_modifier"] = int(relation_sensitive)
    features["candidate_qa_same_sentence"] = int(same_sentence)

    features["qa_effective_agreement"] = max(
        features["qa_normalized_agreement"], float(rank_agreement)
    )
    features["qa_effective_numeric_conflict"] = int(
        features["qa_numeric_conflict"] == 1 and rank_agreement == 0
    )
    features["qa_safe_high_agreement"] = int(
        wrong_numeric_type == 0
        and (
            features["qa_numeric_exact_match"] == 1
            or rank_agreement == 1
            or (features["qa_effective_agreement"] >= 0.95 and relation_sensitive == 0)
        )
    )
    return features


In [ ]:
%%writefile hallu/route_a.py
"""Route A -- contextual rows (plans.md S2), ported from heuristic_qa.ipynb.

Retrieval (question-only, hybrid TF-IDF) -> Bengali extractive QA -> base +
QA-comparison features -> StandardScaler + balanced LogisticRegression.

The QA model (`sagorsarker/mbert-bengali-tydiqa-qa`) is optional: when
transformers/weights are unavailable the pipeline falls back to an empty QA
answer and relies on the base lexical/numeric features.  Public entry point is
``predict_context_rows``.
"""

import os

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from rapidfuzz.fuzz import ratio

from .normalize import (
    clean_context,
    extract_numbers,
    normalize_answer_text,
    normalize_text,
    split_sentences,
    tokenize,
)
from .features import (
    answer_type_mismatch,
    build_comparison_features,
)

QA_MODEL_NAME = "sagorsarker/mbert-bengali-tydiqa-qa"


# ---------------------------------------------------------------------------
# Evidence retrieval (cells 10-13)
# ---------------------------------------------------------------------------
def build_sentence_windows(context, window_size=2, stride=1, max_words=130):
    sentences = split_sentences(context)
    if len(sentences) == 1:
        return sentences
    chunks = []
    for start in range(0, len(sentences), stride):
        selected, word_count = [], 0
        for sentence in sentences[start:start + window_size]:
            sentence_words = sentence.split()
            if selected and word_count + len(sentence_words) > max_words:
                break
            selected.append(sentence)
            word_count += len(sentence_words)
        if selected:
            chunk = " ".join(selected)
            if chunk not in chunks:
                chunks.append(chunk)
    for sentence in sentences:
        if sentence not in chunks:
            chunks.append(sentence)
    return chunks


def _safe_cosine_tfidf(chunks, query, analyzer, ngram_range):
    if len(chunks) == 1:
        return np.array([1.0], dtype=float)
    try:
        vectorizer = TfidfVectorizer(
            analyzer=analyzer, ngram_range=ngram_range,
            min_df=1, sublinear_tf=True, lowercase=False,
        )
        matrix = vectorizer.fit_transform(chunks + [query])
        return cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    except ValueError:
        return np.zeros(len(chunks), dtype=float)


def _prompt_chunk_token_overlap(prompt, chunk):
    prompt_tokens = set(tokenize(prompt))
    if not prompt_tokens:
        return 0.0
    return len(prompt_tokens & set(tokenize(chunk))) / len(prompt_tokens)


def _prompt_chunk_number_overlap(prompt, chunk):
    prompt_numbers = extract_numbers(prompt)
    if not prompt_numbers:
        return 0.0
    return len(prompt_numbers & extract_numbers(chunk)) / len(prompt_numbers)


def _chunk_similarity(a, b):
    ta, tb = set(tokenize(a)), set(tokenize(b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def _select_diverse_chunks(chunks, relevance_scores, top_k=5, diversity_weight=0.15):
    if not chunks:
        return [], []
    remaining, selected = list(range(len(chunks))), []
    while remaining and len(selected) < top_k:
        best_index, best_score = None, -np.inf
        for index in remaining:
            redundancy = max(
                (_chunk_similarity(chunks[index], chunks[s]) for s in selected),
                default=0.0,
            )
            mmr = relevance_scores[index] - diversity_weight * redundancy
            if mmr > best_score:
                best_score, best_index = mmr, index
        selected.append(best_index)
        remaining.remove(best_index)
    return (
        [chunks[i] for i in selected],
        [float(relevance_scores[i]) for i in selected],
    )


def retrieve_evidence(context, prompt, top_k=5, window_size=2, short_context_words=220):
    context = normalize_text(context)
    if len(context.split()) <= short_context_words:
        return [context], [1.0]
    chunks = build_sentence_windows(context, window_size=window_size, stride=1, max_words=130)
    if len(chunks) == 1:
        return chunks, [1.0]
    normalized_prompt = normalize_text(prompt)
    normalized_chunks = [normalize_text(c) for c in chunks]
    char_scores = _safe_cosine_tfidf(normalized_chunks, normalized_prompt, "char_wb", (2, 5))
    word_scores = _safe_cosine_tfidf(normalized_chunks, normalized_prompt, "word", (1, 2))
    token_overlap = np.array(
        [_prompt_chunk_token_overlap(normalized_prompt, c) for c in normalized_chunks],
        dtype=float,
    )
    number_overlap = np.array(
        [_prompt_chunk_number_overlap(normalized_prompt, c) for c in normalized_chunks],
        dtype=float,
    )
    final_scores = (
        0.45 * char_scores + 0.30 * word_scores
        + 0.20 * token_overlap + 0.05 * number_overlap
    )
    return _select_diverse_chunks(
        chunks, final_scores, top_k=min(top_k, len(chunks)), diversity_weight=0.15
    )


# ---------------------------------------------------------------------------
# Base (context) features (cell 17)
# ---------------------------------------------------------------------------
def _token_f1_local(a, b):
    from .normalize import token_f1
    return token_f1(a, b)


def _lexical_features(response, evidence):
    rt = tokenize(response)
    if not rt:
        return {
            "context_exact_token_overlap": 0.0,
            "context_soft_token_overlap": 0.0,
            "context_exact_response_span": 0.0,
        }
    et = tokenize(evidence)
    eset = set(et)
    exact = sum(t in eset for t in rt)
    soft = sum(int(max((ratio(t, x) for x in et), default=0) >= 85) for t in rt)
    rn, en = normalize_answer_text(response), normalize_answer_text(evidence)
    return {
        "context_exact_token_overlap": exact / len(rt),
        "context_soft_token_overlap": soft / len(rt),
        "context_exact_response_span": float(bool(rn) and rn in en),
    }


def _numeric_context_features(response, evidence, full_context):
    rn = extract_numbers(response)
    en = extract_numbers(evidence)
    cn = extract_numbers(full_context)
    has = bool(rn)
    return {
        "response_has_number": int(has),
        "evidence_has_number": int(bool(en)),
        "context_has_number": int(bool(cn)),
        "number_supported_evidence": int(has and rn.issubset(en)),
        "number_supported_context": int(has and rn.issubset(cn)),
        "unsupported_evidence_numbers": len(rn - en) if has else 0,
        "unsupported_context_numbers": len(rn - cn) if has else 0,
    }


def _base_features_row(row, chunks, scores):
    evidence = " ".join(chunks)
    features = {}
    features.update(_lexical_features(row["response_bn"], evidence))
    features.update(_numeric_context_features(row["response_bn"], evidence, row["context"]))
    features["answer_type_mismatch"] = answer_type_mismatch(row["prompt_bn"], row["response_bn"])
    features["retrieval_top_score"] = scores[0] if scores else 0.0
    features["retrieval_margin"] = scores[0] - scores[1] if len(scores) >= 2 else 0.0
    features["response_token_count"] = len(tokenize(row["response_bn"]))
    features["context_chunk_count"] = len(chunks)
    return features


# ---------------------------------------------------------------------------
# Optional Bengali extractive QA (cells 19-21)
# ---------------------------------------------------------------------------
class _QAEngine:
    """Lazy wrapper around the extractive QA model; no-op if unavailable."""

    def __init__(self, model_name=QA_MODEL_NAME, device=None):
        self.available = False
        self.model = self.tokenizer = None
        try:
            import torch
            from transformers import AutoModelForQuestionAnswering, AutoTokenizer

            self.torch = torch
            self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = (
                AutoModelForQuestionAnswering.from_pretrained(model_name).to(self.device).eval()
            )
            self.available = True
        except Exception as exc:  # noqa: BLE001 -- degrade to lexical mode
            print(f"[route_a] QA model unavailable ({type(exc).__name__}); lexical fallback")

    def _best_span(self, context, offsets, sequence_ids, start_logits, end_logits,
                   max_answer_tokens=40, top_candidates=20):
        torch = self.torch
        ctx_idx = [i for i, s in enumerate(sequence_ids) if s == 1]
        if not ctx_idx:
            return {"answer": "", "raw_score": 0.0, "logit_score": -1e9}
        index_set = set(ctx_idx)
        vs, ve = start_logits.clone(), end_logits.clone()
        for i in range(len(sequence_ids)):
            if i not in index_set:
                vs[i] = ve[i] = -1e9
        top_starts = torch.topk(vs, k=min(top_candidates, len(ctx_idx))).indices.tolist()
        top_ends = torch.topk(ve, k=min(top_candidates, len(ctx_idx))).indices.tolist()
        sp, ep = torch.softmax(start_logits, -1), torch.softmax(end_logits, -1)
        best = {"answer": "", "raw_score": 0.0, "logit_score": -1e9}
        for s in top_starts:
            for e in top_ends:
                if e < s or e - s + 1 > max_answer_tokens:
                    continue
                if s not in index_set or e not in index_set:
                    continue
                answer = context[offsets[s][0]:offsets[e][1]].strip()
                if not answer:
                    continue
                logit_score = float(start_logits[s] + end_logits[e])
                if logit_score > best["logit_score"]:
                    best = {
                        "answer": answer,
                        "raw_score": float(sp[s] * ep[e]),
                        "logit_score": logit_score,
                    }
        return best

    def run(self, records, evidence, batch_size=24, max_length=512):
        empty = {
            "qa_answer": "", "qa_raw_score": 0.0, "qa_logit_score": -1e9,
            "qa_joint_score": -1e9, "qa_retrieval_score": 0.0, "qa_evidence": "",
        }
        if not self.available:
            return [dict(empty) for _ in records]

        torch = self.torch
        pairs = []
        for ri, (row, info) in enumerate(zip(records, evidence)):
            for chunk, score in zip(info["chunks"], info["scores"]):
                pairs.append({"row_index": ri, "question": row["prompt_bn"],
                              "context": chunk, "retrieval_score": float(score)})
        best_by_row = [dict(empty) for _ in records]
        with torch.inference_mode():
            for start in range(0, len(pairs), batch_size):
                batch = pairs[start:start + batch_size]
                enc = self.tokenizer(
                    [x["question"] for x in batch], [x["context"] for x in batch],
                    padding=True, truncation="only_second", max_length=max_length,
                    return_offsets_mapping=True, return_tensors="pt",
                )
                offsets = enc.pop("offset_mapping")
                seq_ids = [enc.sequence_ids(i) for i in range(len(batch))]
                outputs = self.model(**{k: v.to(self.device) for k, v in enc.items()})
                for i, item in enumerate(batch):
                    span = self._best_span(
                        item["context"], offsets[i].tolist(), seq_ids[i],
                        outputs.start_logits[i].detach().cpu(),
                        outputs.end_logits[i].detach().cpu(),
                    )
                    joint = span["logit_score"] + np.log(max(item["retrieval_score"], 0.05))
                    ri = item["row_index"]
                    if joint > best_by_row[ri]["qa_joint_score"]:
                        best_by_row[ri] = {
                            "qa_answer": span["answer"],
                            "qa_raw_score": span["raw_score"],
                            "qa_logit_score": span["logit_score"],
                            "qa_joint_score": float(joint),
                            "qa_retrieval_score": item["retrieval_score"],
                            "qa_evidence": item["context"],
                        }
        return best_by_row


# ---------------------------------------------------------------------------
# Feature-matrix builder
# ---------------------------------------------------------------------------
def _qa_score_features(result):
    raw = float(result.get("qa_raw_score", 0.0))
    return {
        "qa_raw_score": raw,
        "qa_logit_score": float(result.get("qa_logit_score", 0.0)),
        "qa_joint_score": float(result.get("qa_joint_score", 0.0)),
        "qa_retrieval_score": float(result.get("qa_retrieval_score", 0.0)),
        "qa_answer_token_count": len(tokenize(result.get("qa_answer", ""))),
    }


def build_feature_frame(df, qa_engine=None, batch_size=24):
    """Build the route-A feature matrix for a cleaned context DataFrame."""
    records = df.to_dict("records")
    evidence = []
    for row in records:
        chunks, scores = retrieve_evidence(row["context"], row["prompt_bn"])
        evidence.append({"chunks": chunks, "scores": scores})

    qa_engine = qa_engine if qa_engine is not None else _QAEngine()
    qa_results = qa_engine.run(records, evidence, batch_size=batch_size)

    rows = []
    for row, info, result in zip(records, evidence, qa_results):
        feats = _base_features_row(row, info["chunks"], info["scores"])
        comp = build_comparison_features(
            prompt=row["prompt_bn"], response=row["response_bn"],
            gold_answer=result["qa_answer"], context=row["context"],
        )
        feats.update(comp)
        feats.update(_qa_score_features(result))
        feats["qa_confident_agreement"] = (
            feats["qa_raw_score"] * feats["qa_effective_agreement"]
        )
        feats["qa_confident_numeric_conflict"] = (
            feats["qa_raw_score"] * feats["qa_effective_numeric_conflict"]
        )
        rows.append(feats)

    feature_df = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return feature_df, qa_results


def clean_context_frame(df):
    """Keep rows with a real context; string-cast the text columns."""
    out = df.copy()
    out["_has_ctx"] = out["context"].map(lambda x: len(clean_context(x)) > 0)
    out = out[out["_has_ctx"]].drop(columns="_has_ctx")
    for c in ("context", "prompt_bn", "response_bn"):
        out[c] = out[c].fillna("").astype(str)
    return out.reset_index(drop=True)


def make_calibrator(seed=42):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=0.5, class_weight="balanced", max_iter=3000, random_state=seed)),
    ])


def predict_context_rows(df_ctx, train_df, threshold=0.5, qa_engine=None,
                         return_details=False, seed=42):
    """Predict labels for context rows (plans.md S2 deliverable).

    Fits the calibrator on labelled context rows in ``train_df`` and applies it
    to ``df_ctx``.  Returns a DataFrame ``[id, label]`` (original ``id`` column
    required in ``df_ctx``).  A single shared QA engine can be passed to avoid
    reloading the model.
    """
    qa_engine = qa_engine if qa_engine is not None else _QAEngine()

    train_clean = clean_context_frame(train_df)
    train_clean = train_clean[train_clean["label"].isin([0, 1])].reset_index(drop=True)
    X_train, _ = build_feature_frame(train_clean, qa_engine=qa_engine)
    y_train = train_clean["label"].to_numpy(dtype=int)
    feature_columns = X_train.columns.tolist()

    model = make_calibrator(seed=seed)
    model.fit(X_train, y_train)

    test_clean = clean_context_frame(df_ctx)
    X_test, qa_results = build_feature_frame(test_clean, qa_engine=qa_engine)
    X_test = X_test.reindex(columns=feature_columns, fill_value=0.0)
    proba = model.predict_proba(X_test)[:, 1]
    labels = (proba >= threshold).astype(int)

    result = pd.DataFrame({"id": test_clean["id"].to_numpy(), "label": labels})
    if return_details:
        details = test_clean.copy()
        details["probability_faithful"] = proba
        details["prediction"] = labels
        return result, details, model, feature_columns
    return result


In [ ]:
%%writefile hallu/indices.py
"""KB indices for route B (plans.md S4.1), built once and cached to disk.

Artifacts (all keyed on ``kb_id``):
    (a) inverted index over KB questions   -- token -> set(kb_id)
    (b) inverted index over KB answers     -- token -> set(kb_id)
    (c) NER inverted index                 -- entity -> set(kb_id)  (gazetteer/
        regex fallback so it works fully offline; a transformer NER tagger can
        replace `extract_entities` when weights are attached)
    (d) BM25 index over "question + answer" tokens
    (e) LaBSE question embeddings (optional; float32 matrix + kb_id row map)

Building is guarded by a cache check (`build_indices`).  LaBSE is skipped
gracefully when sentence-transformers/weights are unavailable, leaving route B
in pure-lexical mode.
"""

import os
import pickle
import re
from collections import defaultdict

import numpy as np

from .normalize import BN2ASCII, normalize_answer_text, normalize_text


# ---------------------------------------------------------------------------
# Tokenisation for indexing / BM25
# ---------------------------------------------------------------------------
_STOP_INDEX = set(
    "এর এই এবং কি কী কে ও যে যা কোন কত হয় ছিল করে থেকে জন্য নামে".split()
)


def index_tokens(text, min_len=2):
    """Word tokens over the canonical form, light stoplist, digit-folded."""
    toks = normalize_answer_text(text).split()
    return [t for t in toks if len(t) >= min_len and t not in _STOP_INDEX]


# ---------------------------------------------------------------------------
# (c) Entity extraction -- offline gazetteer/regex fallback
# ---------------------------------------------------------------------------
# Proper-noun-ish heuristic: Bengali has no case, so we anchor on numbers,
# years/dates, and multi-char content tokens that are not common function words.
_YEAR_RE = re.compile(r"\b(1[0-9]{3}|2[0-9]{3})\b")
_NUM_RE = re.compile(r"\b\d+(?:\.\d+)?\b")
_ENTITY_STOP = _STOP_INDEX | set(
    "সালে সাল বছর মধ্যে সঙ্গে সাথে পরে আগে তখন এখন কেন কবে কোথায় কারা "
    "একটি একজন প্রথম শেষ".split()
)


def extract_entities(text):
    """Return a set of normalized entity surface forms.

    Fallback strategy: numbers + 4-digit years + content tokens of length >= 4
    (proper nouns in Bengali tend to be longer and are rarely stopwords).  This
    is deliberately high-recall; the NER index is one recall signal among many.
    """
    folded = normalize_text(text).translate(BN2ASCII)
    ents = set()
    ents.update(_YEAR_RE.findall(folded))
    ents.update(_NUM_RE.findall(folded))
    for tok in normalize_answer_text(text).split():
        if len(tok) >= 4 and tok not in _ENTITY_STOP and not tok.isdigit():
            ents.add(tok)
    return ents


# ---------------------------------------------------------------------------
# Build / cache
# ---------------------------------------------------------------------------
def _build_inverted(texts, kb_ids):
    inv = defaultdict(set)
    for kb_id, text in zip(kb_ids, texts):
        for tok in set(index_tokens(text)):
            inv[tok].add(int(kb_id))
    return {k: v for k, v in inv.items()}


def _build_ner_index(questions, answers, kb_ids):
    inv = defaultdict(set)
    for kb_id, q, a in zip(kb_ids, questions, answers):
        for ent in extract_entities(q) | extract_entities(a):
            inv[ent].add(int(kb_id))
    return {k: v for k, v in inv.items()}


class KBIndices:
    """Container for the five KB artifacts + query helpers."""

    def __init__(self, kb_ids, q_tokens, inv_q, inv_a, inv_ner, bm25,
                 labse=None, labse_ids=None):
        self.kb_ids = np.asarray(kb_ids)
        self.q_tokens = q_tokens          # list[list[str]] aligned to kb rows
        self.inv_q = inv_q
        self.inv_a = inv_a
        self.inv_ner = inv_ner
        self.bm25 = bm25                  # rank_bm25.BM25Okapi over q+a tokens
        self.labse = labse                # (N, d) float32, L2-normalized, or None
        self.labse_ids = labse_ids        # kb_id per labse row, or None

    @property
    def has_labse(self):
        return self.labse is not None


def build_indices(kb, cache_dir, use_labse=True, labse_model="sentence-transformers/LaBSE",
                  labse_batch_size=256, verbose=True):
    """Build (or load) all KB indices for route B.  Guarded by cache existence."""
    os.makedirs(cache_dir, exist_ok=True)
    lex_path = os.path.join(cache_dir, "kb_lexical_indices.pkl")
    labse_path = os.path.join(cache_dir, "labse_kb.npy")
    labse_ids_path = os.path.join(cache_dir, "labse_kb_ids.npy")

    kb_ids = kb["kb_id"].to_numpy()

    if os.path.exists(lex_path):
        if verbose:
            print(f"[indices] loading lexical indices from {lex_path}")
        with open(lex_path, "rb") as fh:
            store = pickle.load(fh)
        q_tokens = store["q_tokens"]
        inv_q, inv_a, inv_ner = store["inv_q"], store["inv_a"], store["inv_ner"]
        bm25 = store["bm25"]
    else:
        from rank_bm25 import BM25Okapi

        if verbose:
            print("[indices] tokenizing KB ...")
        questions = kb["prompt_bn"].tolist()
        answers = kb["response_bn"].tolist()
        q_tokens = [index_tokens(q) for q in questions]
        a_tokens = [index_tokens(a) for a in answers]

        if verbose:
            print("[indices] building inverted (Q), inverted (A), NER indices ...")
        inv_q = _build_inverted(questions, kb_ids)
        inv_a = _build_inverted(answers, kb_ids)
        inv_ner = _build_ner_index(questions, answers, kb_ids)

        if verbose:
            print("[indices] building BM25 over question+answer ...")
        corpus = [qt + at for qt, at in zip(q_tokens, a_tokens)]
        bm25 = BM25Okapi(corpus)

        with open(lex_path, "wb") as fh:
            pickle.dump(
                {"q_tokens": q_tokens, "inv_q": inv_q, "inv_a": inv_a,
                 "inv_ner": inv_ner, "bm25": bm25},
                fh, protocol=pickle.HIGHEST_PROTOCOL,
            )
        if verbose:
            print(f"[indices] cached lexical indices -> {lex_path}")

    labse, labse_ids = None, None
    if use_labse:
        if os.path.exists(labse_path) and os.path.exists(labse_ids_path):
            if verbose:
                print(f"[indices] loading LaBSE matrix from {labse_path}")
            labse = np.load(labse_path)
            labse_ids = np.load(labse_ids_path)
        else:
            labse, labse_ids = _encode_labse(
                kb, labse_model, labse_batch_size, kb_ids, verbose
            )
            if labse is not None:
                np.save(labse_path, labse)
                np.save(labse_ids_path, labse_ids)
                if verbose:
                    print(f"[indices] cached LaBSE matrix -> {labse_path}")

    return KBIndices(kb_ids, q_tokens, inv_q, inv_a, inv_ner, bm25, labse, labse_ids)


def _encode_labse(kb, model_name, batch_size, kb_ids, verbose):
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:  # noqa: BLE001
        if verbose:
            print(f"[indices] LaBSE unavailable ({type(exc).__name__}); lexical-only mode")
        return None, None
    try:
        model = SentenceTransformer(model_name)
        if verbose:
            print(f"[indices] encoding {len(kb)} KB questions with LaBSE ...")
        emb = model.encode(
            kb["prompt_bn"].tolist(), batch_size=batch_size,
            show_progress_bar=verbose, convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)
        return emb, np.asarray(kb_ids)
    except Exception as exc:  # noqa: BLE001
        if verbose:
            print(f"[indices] LaBSE encode failed ({type(exc).__name__}); lexical-only mode")
        return None, None


In [ ]:
%%writefile hallu/route_b.py
"""Route B -- no-context factual verifier (plans.md S4.2-4.4).

For each no-context test row we manufacture evidence by retrieving the best
matching (question, gold_answer) unit from the KB, then decide faithfulness by
comparing the test response against that gold answer.

Retrieval fuses four normalized signals -- BM25, rapidfuzz, NER-overlap, LaBSE
cosine -- over a union-recall candidate pool.  The faithfulness decision reuses
the shared comparison features (features.build_comparison_features) plus the
retrieval signals, fed to a StandardScaler + balanced LogisticRegression
calibrator.  Low-confidence rows abstain to a prior-based fallback (S4.4).
"""

import json
import os

import numpy as np
import pandas as pd

from rapidfuzz.fuzz import ratio, token_set_ratio

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .features import build_comparison_features
from .indices import extract_entities, index_tokens
from .normalize import clean_context, normalize_answer_text

DEFAULT_WEIGHTS = {"bm25": 0.30, "fuzz": 0.25, "ner": 0.15, "labse": 0.30}

# Retrieval signals appended to the comparison features for the calibrator.
RETRIEVAL_FEATURES = [
    "S_best", "bm25_hat", "fuzz_hat", "ner_hat", "labse_hat",
    "bm25_raw", "labse_raw", "fuzz_q_raw", "fuzz_a_raw", "ner_shared",
]


def _minmax(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-9:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)


class RouteB:
    def __init__(self, kb, indices, weights=None, labse_encoder=None,
                 candidate_cap=300, bm25_topn=100, labse_topn=100, fuzzy_topn=20,
                 abstain_threshold=0.15):
        self.kb = kb.reset_index(drop=True)
        self.idx = indices
        self.weights = dict(weights or DEFAULT_WEIGHTS)
        self.candidate_cap = candidate_cap
        self.bm25_topn = bm25_topn
        self.labse_topn = labse_topn
        self.fuzzy_topn = fuzzy_topn
        self.abstain_threshold = abstain_threshold

        # kb_id -> row position (kb_id equals row index by construction, but be safe).
        self._pos = {int(k): i for i, k in enumerate(self.kb["kb_id"].to_numpy())}
        self._q_text = self.kb["prompt_bn"].tolist()
        self._a_text = self.kb["response_bn"].tolist()
        # Pre-normalize KB text (q_norm/a_norm come from the KB loader) and cache
        # per-row entity sets lazily -- both dominate the per-candidate cost.
        self._q_norm = self.kb["q_norm"].tolist()
        self._a_norm = self.kb["a_norm"].tolist()
        self._ent_cache = {}
        self._labse_pos = None
        if self.idx.has_labse:
            self._labse_pos = {int(k): i for i, k in enumerate(self.idx.labse_ids)}

        self.labse_encoder = labse_encoder  # callable(list[str]) -> (n, d) L2-normed
        self.calibrator = None
        self.feature_columns = None
        self.threshold = 0.5
        self.fallback_label = 1

    # ------------------------------------------------------------------ retrieval
    def _candidate_ids(self, q_tokens, a_tokens, entities, bm25_scores, labse_scores):
        cands = set()
        for tok in q_tokens:
            cands |= self.idx.inv_q.get(tok, set())
        for tok in a_tokens:
            cands |= self.idx.inv_a.get(tok, set())
        for ent in entities:
            cands |= self.idx.inv_ner.get(ent, set())

        if bm25_scores is not None and self.bm25_topn:
            top = np.argpartition(-bm25_scores, min(self.bm25_topn, len(bm25_scores) - 1))
            cands |= {int(self.idx.kb_ids[i]) for i in top[: self.bm25_topn]}
        if labse_scores is not None and self.labse_topn:
            top = np.argpartition(-labse_scores, min(self.labse_topn, len(labse_scores) - 1))
            cands |= {int(self.idx.labse_ids[i]) for i in top[: self.labse_topn]}

        cands = list(cands)
        # Cap by BM25 to keep the expensive scoring bounded.
        if len(cands) > self.candidate_cap and bm25_scores is not None:
            scored = sorted(
                cands, key=lambda cid: bm25_scores[self._pos[cid]], reverse=True
            )
            cands = scored[: self.candidate_cap]
        return cands

    def _cand_entities(self, pos):
        ents = self._ent_cache.get(pos)
        if ents is None:
            ents = extract_entities(self._q_text[pos]) | extract_entities(self._a_text[pos])
            self._ent_cache[pos] = ents
        return ents

    def retrieve(self, prompt, response):
        """Return the best-matching KB unit and its (normalized) retrieval signals."""
        q_tokens = index_tokens(prompt)
        a_tokens = index_tokens(response)
        entities = extract_entities(prompt) | extract_entities(response)
        query_tokens = q_tokens + a_tokens

        bm25_scores = self.idx.bm25.get_scores(query_tokens) if query_tokens else None

        labse_scores = None
        q_vec = None
        if self.idx.has_labse and self.labse_encoder is not None:
            q_vec = self.labse_encoder([prompt])[0]
            labse_scores = self.idx.labse @ q_vec

        cand_ids = self._candidate_ids(q_tokens, a_tokens, entities, bm25_scores, labse_scores)
        if not cand_ids:
            return None

        # Per-candidate raw signals.
        bm25_raw, labse_raw, fuzz_q, fuzz_a, ner_shared = [], [], [], [], []
        p_norm = normalize_answer_text(prompt)
        r_norm = normalize_answer_text(response)
        for cid in cand_ids:
            pos = self._pos[cid]
            bm25_raw.append(bm25_scores[pos] if bm25_scores is not None else 0.0)
            if labse_scores is not None and cid in self._labse_pos:
                labse_raw.append(float(labse_scores[self._labse_pos[cid]]))
            else:
                labse_raw.append(0.0)
            fuzz_q.append(token_set_ratio(p_norm, self._q_norm[pos]) / 100.0)
            fuzz_a.append(ratio(r_norm, self._a_norm[pos]) / 100.0)
            shared = len(entities & self._cand_entities(pos))
            denom = len(entities) if entities else 1
            ner_shared.append(shared / denom)

        bm25_raw = np.asarray(bm25_raw)
        labse_raw = np.asarray(labse_raw)
        fuzz_q = np.asarray(fuzz_q)
        fuzz_a = np.asarray(fuzz_a)
        ner_shared = np.asarray(ner_shared)
        fuzz_combined = np.maximum(fuzz_q, fuzz_a)

        # Per-row normalization so signals share a [0,1] scale before fusion.
        bm25_hat = _minmax(bm25_raw)
        fuzz_hat = _minmax(fuzz_combined)
        ner_hat = _minmax(ner_shared)
        labse_hat = _minmax(labse_raw) if labse_scores is not None else np.zeros(len(cand_ids))

        w = self.weights
        fused = (
            w["bm25"] * bm25_hat + w["fuzz"] * fuzz_hat
            + w["ner"] * ner_hat + w["labse"] * labse_hat
        )
        best = int(np.argmax(fused))
        cid = cand_ids[best]
        pos = self._pos[cid]
        return {
            "kb_id": cid,
            "kb_question": self._q_text[pos],
            "kb_gold_answer": self._a_text[pos],
            "S_best": float(fused[best]),
            "bm25_hat": float(bm25_hat[best]),
            "fuzz_hat": float(fuzz_hat[best]),
            "ner_hat": float(ner_hat[best]),
            "labse_hat": float(labse_hat[best]),
            "bm25_raw": float(bm25_raw[best]),
            "labse_raw": float(labse_raw[best]),
            "fuzz_q_raw": float(fuzz_q[best]),
            "fuzz_a_raw": float(fuzz_a[best]),
            "ner_shared": float(ner_shared[best]),
        }

    # ------------------------------------------------------------------ features
    def build_feature_frame(self, df):
        """Feature matrix + retrieval detail for a cleaned no-context DataFrame."""
        rows, details = [], []
        for row in df.to_dict("records"):
            match = self.retrieve(row["prompt_bn"], row["response_bn"])
            if match is None:
                feats = build_comparison_features(row["prompt_bn"], row["response_bn"], "")
                for name in RETRIEVAL_FEATURES:
                    feats[name] = 0.0
                feats["retrieval_missing"] = 1
                details.append({"kb_id": -1, "S_best": 0.0, "kb_gold_answer": ""})
            else:
                feats = build_comparison_features(
                    row["prompt_bn"], row["response_bn"], match["kb_gold_answer"]
                )
                for name in RETRIEVAL_FEATURES:
                    feats[name] = match[name]
                feats["retrieval_missing"] = 0
                details.append(match)
            rows.append(feats)
        feature_df = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return feature_df, pd.DataFrame(details)

    # ------------------------------------------------------------------ fit / predict
    def fit(self, train_df, threshold=None, seed=42):
        """Fit the route-B calibrator on labelled no-context sample rows."""
        clean = _clean_noctx(train_df)
        clean = clean[clean["label"].isin([0, 1])].reset_index(drop=True)
        X, details = self.build_feature_frame(clean)
        y = clean["label"].to_numpy(dtype=int)
        self.feature_columns = X.columns.tolist()

        self.calibrator = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=0.5, class_weight="balanced", max_iter=3000, random_state=seed)),
        ])
        self.calibrator.fit(X, y)

        # Prior-based fallback default = majority label of low-confidence rows.
        low = details["S_best"].to_numpy() < self.abstain_threshold
        if low.any():
            self.fallback_label = int(round(y[low].mean()))
        else:
            self.fallback_label = int(round(y.mean()))

        if threshold is not None:
            self.threshold = threshold
        return self

    def predict(self, df, return_details=False):
        clean = _clean_noctx(df)
        X, details = self.build_feature_frame(clean)
        X = X.reindex(columns=self.feature_columns, fill_value=0.0)
        proba = self.calibrator.predict_proba(X)[:, 1]
        labels = (proba >= self.threshold).astype(int)

        # Abstain -> fallback for rows with no trustworthy match (S4.4).
        abstain = details["S_best"].to_numpy() < self.abstain_threshold
        labels[abstain] = self.fallback_label
        path = np.where(abstain, "fallback", "verifier")

        result = pd.DataFrame({"id": clean["id"].to_numpy(), "label": labels})
        if return_details:
            det = clean.copy()
            det["probability_faithful"] = proba
            det["prediction"] = labels
            det["path"] = path
            det["S_best"] = details["S_best"].to_numpy()
            det["kb_gold_answer"] = details["kb_gold_answer"].to_numpy()
            return result, det
        return result

    # ------------------------------------------------------------------ persistence
    def save(self, out_dir):
        import joblib

        os.makedirs(out_dir, exist_ok=True)
        joblib.dump(self.calibrator, os.path.join(out_dir, "route_b_calibrator.joblib"))
        with open(os.path.join(out_dir, "route_b_config.json"), "w", encoding="utf-8") as fh:
            json.dump({
                "weights": self.weights,
                "threshold": float(self.threshold),
                "abstain_threshold": float(self.abstain_threshold),
                "fallback_label": int(self.fallback_label),
                "feature_columns": self.feature_columns,
            }, fh, ensure_ascii=False, indent=2)


def _clean_noctx(df):
    out = df.copy()
    out["_ctx"] = out["context"].map(lambda x: len(clean_context(x)) > 0)
    out = out[~out["_ctx"]].drop(columns="_ctx")
    for c in ("prompt_bn", "response_bn"):
        out[c] = out[c].fillna("").astype(str)
    return out.reset_index(drop=True)


def make_labse_encoder(model_name="sentence-transformers/LaBSE"):
    """Return a callable encoding a list of strings to L2-normalized vectors, or None."""
    try:
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(model_name)

        def encode(texts):
            return model.encode(
                list(texts), convert_to_numpy=True, normalize_embeddings=True
            ).astype(np.float32)

        return encode
    except Exception as exc:  # noqa: BLE001
        print(f"[route_b] LaBSE encoder unavailable ({type(exc).__name__}); lexical-only")
        return None


In [ ]:
%%writefile hallu/validate.py
"""Validation & tuning on the labelled sample split (plans.md S5).

The sample split (`dataset samples.json`, 299 rows) is the only labelled data.
We split it by ``has_context`` and validate each route independently, then
report the combined macro-F1.

Route B tuning is grouped by normalized prompt (avoid leaking near-duplicate
questions across folds) and coordinate-searches the fusion weights + decision
threshold for macro-F1.
"""

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .normalize import has_context, normalize_answer_text
from .route_b import RETRIEVAL_FEATURES, RouteB


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


def _grouped_folds(y, groups, n_splits, seed):
    n_splits = min(n_splits, int(pd.Series(y).value_counts().min()), len(np.unique(groups)))
    n_splits = max(2, n_splits)
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(splitter.split(np.zeros(len(y)), y, groups))


def _best_threshold(y_true, proba):
    best_t, best_f = 0.5, -1.0
    for t in np.arange(0.20, 0.81, 0.01):
        f = macro_f1(y_true, (proba >= t).astype(int))
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t, best_f


def tune_route_b(route_b, sample_df, weight_grid=None, n_splits=5, seed=42, verbose=True):
    """Coordinate-search fusion weights + threshold on the no-context sample rows.

    Features are recomputed per weight setting (retrieval depends on the weights),
    but the expensive per-candidate signals are cached inside ``route_b``, so the
    search stays cheap.  Returns (best_weights, best_threshold, oof_macro_f1).
    """
    noctx = sample_df[~sample_df["context"].map(has_context)].reset_index(drop=True)
    noctx = noctx[noctx["label"].isin([0, 1])].reset_index(drop=True)
    y = noctx["label"].to_numpy(dtype=int)
    groups = noctx["prompt_bn"].map(normalize_answer_text).to_numpy()
    folds = _grouped_folds(y, groups, n_splits, seed)

    if weight_grid is None:
        # Candidate weight vectors (bm25, fuzz, ner, labse); labse=0 in lexical mode.
        weight_grid = [
            {"bm25": 0.30, "fuzz": 0.25, "ner": 0.15, "labse": 0.30},
            {"bm25": 0.40, "fuzz": 0.30, "ner": 0.15, "labse": 0.15},
            {"bm25": 0.35, "fuzz": 0.35, "ner": 0.20, "labse": 0.10},
            {"bm25": 0.50, "fuzz": 0.30, "ner": 0.20, "labse": 0.00},
            {"bm25": 0.45, "fuzz": 0.40, "ner": 0.15, "labse": 0.00},
            {"bm25": 0.34, "fuzz": 0.33, "ner": 0.33, "labse": 0.00},
        ]
    if not route_b.idx.has_labse:
        weight_grid = [w for w in weight_grid if w["labse"] == 0.0] or [
            {"bm25": 0.40, "fuzz": 0.35, "ner": 0.25, "labse": 0.0}
        ]

    best = {"macro_f1": -1.0}
    for weights in weight_grid:
        route_b.weights = dict(weights)
        X, details = route_b.build_feature_frame(noctx)
        feature_columns = X.columns.tolist()
        Xv = X.to_numpy(dtype=float)

        oof = np.full(len(y), np.nan)
        for tr, va in folds:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(
                    C=0.5, class_weight="balanced", max_iter=3000, random_state=seed)),
            ])
            model.fit(Xv[tr], y[tr])
            oof[va] = model.predict_proba(Xv[va])[:, 1]

        threshold, f = _best_threshold(y, oof)
        if verbose:
            print(f"  weights={weights} -> thr={threshold:.2f} macro_f1={f:.4f}")
        if f > best["macro_f1"]:
            best = {
                "macro_f1": f, "weights": dict(weights), "threshold": threshold,
                "feature_columns": feature_columns, "oof": oof, "y": y,
            }

    route_b.weights = dict(best["weights"])
    if verbose:
        print(f"[route_b] best: weights={best['weights']} thr={best['threshold']:.2f} "
              f"OOF macro_f1={best['macro_f1']:.4f}")
    return best


In [ ]:
%%writefile hallu/pipeline.py
"""Assembly & submission (plans.md S6).

Routes the test set by ``has_context``, runs route A on context rows and route B
on no-context rows, reassembles predictions in the original test order, and
writes ``submission.csv`` with sanity checks.
"""

import json

import pandas as pd

from .kb import load_kb
from .indices import build_indices
from .normalize import has_context
from .route_a import predict_context_rows, _QAEngine
from .route_b import RouteB, make_labse_encoder
from .validate import tune_route_b


def split_by_context(test_df):
    mask = test_df["context"].map(has_context)
    return test_df[mask].copy(), test_df[~mask].copy()


def assemble_submission(test_df, label_frames, out_path=None):
    """Merge per-route [id, label] frames back into original test order."""
    labels = pd.concat(label_frames, ignore_index=True)
    dup = labels["id"].duplicated().sum()
    if dup:
        raise ValueError(f"{dup} ids predicted by more than one route")

    submission = test_df[["id"]].merge(labels, on="id", how="left")
    missing = int(submission["label"].isna().sum())
    if missing:
        raise ValueError(f"{missing} test ids have no prediction")
    submission["label"] = submission["label"].astype(int)
    if not submission["label"].isin([0, 1]).all():
        raise ValueError("labels outside {0, 1}")
    assert len(submission) == len(test_df) and submission["id"].is_unique

    if out_path:
        submission.to_csv(out_path, index=False)
    return submission


def run(
    test_csv,
    kb_csv,
    sample_json,
    out_path="submission.csv",
    cache_dir="artifacts",
    use_labse=True,
    tune=True,
    verbose=True,
    qa_model=None,
    labse_model=None,
):
    """End-to-end submission run (plans.md S6).

    ``qa_model`` / ``labse_model`` may be local paths (e.g. a Kaggle-attached
    model dataset) or HF hub ids; ``None`` uses the plan defaults.
    """
    test_df = pd.read_csv(test_csv)
    sample_df = pd.DataFrame(_load_json(sample_json))
    df_ctx, df_noctx = split_by_context(test_df)
    if verbose:
        print(f"[pipeline] test={len(test_df)} context={len(df_ctx)} no-context={len(df_noctx)}")

    # ---- Route A (contextual) ------------------------------------------------
    qa_engine = _QAEngine(model_name=qa_model) if qa_model else _QAEngine()
    label_a = predict_context_rows(df_ctx, train_df=sample_df, qa_engine=qa_engine)
    if verbose:
        print(f"[pipeline] route A: {len(label_a)} rows, "
              f"faithful share={label_a['label'].mean():.3f}")

    # ---- Route B (no-context) ------------------------------------------------
    # Leakage guard: drop KB units equal to a sample question (S5).
    kb = load_kb(kb_csv, cache_path=f"{cache_dir}/kb_cache.pkl",
                 sample_questions=sample_df["prompt_bn"].tolist())
    labse_kwargs = {"labse_model": labse_model} if labse_model else {}
    indices = build_indices(kb, cache_dir=cache_dir, use_labse=use_labse,
                            verbose=verbose, **labse_kwargs)
    encoder = None
    if indices.has_labse:
        encoder = make_labse_encoder(labse_model) if labse_model else make_labse_encoder()
    route_b = RouteB(kb, indices, labse_encoder=encoder)

    if tune:
        best = tune_route_b(route_b, sample_df, verbose=verbose)
        route_b.fit(sample_df, threshold=best["threshold"])
    else:
        route_b.fit(sample_df)
    label_b = route_b.predict(df_noctx)
    if verbose:
        print(f"[pipeline] route B: {len(label_b)} rows, "
              f"faithful share={label_b['label'].mean():.3f}")

    # ---- Assemble ------------------------------------------------------------
    submission = assemble_submission(test_df, [label_a, label_b], out_path=out_path)
    if verbose:
        print(f"[pipeline] wrote {out_path}: {len(submission)} rows, "
              f"overall faithful share={submission['label'].mean():.3f}")
    return submission


def _load_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        obj = json.load(fh)
    if isinstance(obj, list):
        return obj
    for key in ("data", "rows", "samples", "train", "examples"):
        if isinstance(obj, dict) and isinstance(obj.get(key), list):
            return obj[key]
    raise ValueError("No list of rows found in JSON")


## 3. Config — auto-detect inputs & optional offline models

In [ ]:
import glob, os, shutil

def _find(*patterns, root="/kaggle/input"):
    for pat in patterns:
        hits = sorted(glob.glob(os.path.join(root, "**", pat), recursive=True))
        if hits:
            return hits[0]
    return None

def _find_model_dir(*name_globs, root="/kaggle/input"):
    """A directory containing config.json whose path matches one of name_globs."""
    for cfg in glob.glob(os.path.join(root, "**", "config.json"), recursive=True):
        d = os.path.dirname(cfg)
        low = d.lower()
        if any(g in low for g in name_globs):
            return d
    return None

TEST_CSV    = _find("test set.csv", "test*set*.csv", "test.csv")
SAMPLE_JSON = _find("dataset samples.json", "*samples*.json", "*sample*.json")
KB_CSV      = _find("combined_bengali_qa.csv", "*combined*bengali*qa*.csv")

# Optional offline models (fall back to HF hub ids -> needs notebook internet ON).
QA_MODEL    = _find_model_dir("tydiqa", "bengali-qa", "bengali_qa") or "sagorsarker/mbert-bengali-tydiqa-qa"
LABSE_MODEL = _find_model_dir("labse")                              or "sentence-transformers/LaBSE"
USE_LABSE   = True          # set False to force pure-lexical Route B

CACHE_DIR = "/kaggle/working/artifacts"
os.makedirs(CACHE_DIR, exist_ok=True)

# If prebuilt indices were attached, copy them into the (writable) cache dir.
for fname in ("kb_lexical_indices.pkl", "labse_kb.npy", "labse_kb_ids.npy", "kb_cache.pkl"):
    src = _find(fname)
    if src and not os.path.exists(os.path.join(CACHE_DIR, fname)):
        shutil.copy(src, os.path.join(CACHE_DIR, fname))
        print("staged prebuilt", fname)

assert TEST_CSV and SAMPLE_JSON and KB_CSV, \
    f"Missing inputs: test={TEST_CSV} sample={SAMPLE_JSON} kb={KB_CSV}"
print("TEST_CSV   :", TEST_CSV)
print("SAMPLE_JSON:", SAMPLE_JSON)
print("KB_CSV     :", KB_CSV)
print("QA_MODEL   :", QA_MODEL)
print("LABSE_MODEL:", LABSE_MODEL, "| USE_LABSE:", USE_LABSE)


## 4. Run the pipeline

Route A fits its calibrator on the labelled **context** sample rows and predicts
the test context rows; Route B tunes fusion weights + threshold on the labelled
**no-context** sample rows, then predicts. Results are reassembled in original
test order.

> First run builds & caches the KB indices (LaBSE encode of ~157k rows is the
> one-time cost). Re-runs reuse `/kaggle/working/artifacts`.

In [ ]:
from hallu.pipeline import run

submission = run(
    test_csv=TEST_CSV,
    kb_csv=KB_CSV,
    sample_json=SAMPLE_JSON,
    out_path="/kaggle/working/submission.csv",
    cache_dir=CACHE_DIR,
    use_labse=USE_LABSE,
    tune=True,
    qa_model=QA_MODEL,
    labse_model=LABSE_MODEL,
    verbose=True,
)


## 5. Sanity check

In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
print("rows:", len(sub))
print("unique ids:", sub["id"].is_unique)
print("labels in {0,1}:", sub["label"].isin([0, 1]).all())
print("label distribution:", sub["label"].value_counts().to_dict())
sub.head()
